# ML-09 — Validation and Research Claim Audit

**Lane:** Refresh / Content Opportunity Scoring

This notebook audits two FlyRank research findings constructively, then audits my Week-5 model. Public-safe language: observed, measured, directional, decision-support.

> Run top-to-bottom in Colab. Requires the FlyRank Hugging Face dataset and `HF_TOKEN` Colab Secret.

## 1. Two paper findings + my methodology questions

### Finding A — The Freshness Multiplier
The report observes a 5.43:1 growth-to-decline ratio for the 31–90 day freshness window and separately reports a large 365+ refreshed-vs-stale difference.

**Methodology question:** What exactly defines the refreshed/freshness label or exposure, and is it determined strictly before the outcome window? Were refreshed pages already different in traffic, age, quality, or intent? Those differences could contribute to the observed gap.

**Validation question:** Is the cohort/stratification design sufficient to separate the refresh decision from the later outcome? If not, the result should be described as an observed association rather than a causal lift.

### Finding B — Refreshing Pages Actually Works
The report states that a held-out test found statistically significant refresh lift in 7 of 9 strata, including a refreshed-vs-stale comparison for 180+ day pages.

**Methodology question:** How is ‘refreshed’ assigned and what is the exact outcome window? I would confirm that the outcome cannot influence the refresh assignment and that the comparison group was eligible under the same rules.

**Validation question:** How were held-out pages/strata selected, and were the 9 strata defined before looking at outcomes? A held-out comparison strengthens evidence but does not by itself remove selection bias.

These are constructive questions about how far the evidence supports the claim; they are not claims that the report is wrong.

In [1]:
print('Paper audit evidence:')
print('Freshness Multiplier: 31–90d = 5.43:1 growth-to-decline ratio (n=18.8K).')
print('Refresh ROI: 7 of 9 strata reported statistically significant refresh lift.')

Paper audit evidence:
Freshness Multiplier: 31–90d = 5.43:1 growth-to-decline ratio (n=18.8K).
Refresh ROI: 7 of 9 strata reported statistically significant refresh lift.


## 2. My model under an honest split (before/after)

**Before:** a naive random row split can put adjacent months and the same clients on both sides. It is a diagnostic, not a deployment estimate.

**After:** train on January–February 2026 and test on March 2026, with the March label coming from April 2026. This respects the March decision moment. Both setups use the same five Week-5 features and Precision@50.

In [2]:
%pip -q install duckdb scikit-learn pandas pyarrow
import duckdb, os, numpy as np, pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

con=duckdb.connect(); token=os.environ.get('HF_TOKEN')
try:
    from google.colab import userdata
    token=token or userdata.get('HF_TOKEN')
except Exception: pass
if not token: raise RuntimeError('HF_TOKEN is missing. Add it to Colab Secrets and enable Notebook access.')
safe_token=token.replace("'","''"); con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")
def rel(m): return f"read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={m}/*.parquet')"
frames=[]
for m in ['2026-01','2026-02','2026-03','2026-04']:
    q=f'''SELECT client_hash_id AS client_id, content_hash_id AS content_id, '{m}' AS month_label, SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks, AVG(gsc_avg_position) AS avg_position, BOOL_OR(ga4_data_available IS TRUE) AS ga4_available FROM {rel(m)} GROUP BY 1,2'''
    frames.append(con.execute(q).df())
df=pd.concat(frames,ignore_index=True).sort_values(['client_id','content_id','month_label'])
df['ctr']=np.where(df.impressions>0,df.clicks/df.impressions,0.0)
df['next_imp']=df.groupby(['client_id','content_id']).impressions.shift(-1)
df['next_month']=df.groupby(['client_id','content_id']).month_label.shift(-1)
expected={'2026-01':'2026-02','2026-02':'2026-03','2026-03':'2026-04'}
df['is_declining_future']=np.where(df.next_month==df.month_label.map(expected),(df.next_imp<0.8*df.impressions).astype(float),np.nan)
features=['impressions','clicks','ctr','avg_position','ga4_available']
d=df.dropna(subset=['is_declining_future']).copy(); d['avg_position']=d.avg_position.fillna(999.0); d['ga4_available']=d.ga4_available.fillna(False).astype(int); d[features]=d[features].replace([np.inf,-np.inf],np.nan).fillna(0)
jan_feb=d[d.month_label.isin(['2026-01','2026-02'])]; march=d[d.month_label=='2026-03']
def p50(y,s): return float(np.mean(np.asarray(y)[np.argsort(-np.asarray(s))[:50]]))
# BEFORE: random row split
Xtr,Xte,ytr,yte=train_test_split(d[features],d.is_declining_future.astype(int),test_size=.20,random_state=42,stratify=d.is_declining_future)
rf0=RandomForestClassifier(n_estimators=100,max_depth=6,min_samples_leaf=20,random_state=42,n_jobs=-1).fit(Xtr,ytr)
random_p50=p50(yte,rf0.predict_proba(Xte)[:,1])
# AFTER: time-aware split
rf1=RandomForestClassifier(n_estimators=100,max_depth=6,min_samples_leaf=20,random_state=42,n_jobs=-1).fit(jan_feb[features],jan_feb.is_declining_future.astype(int))
time_scores=rf1.predict_proba(march[features])[:,1]; time_p50=p50(march.is_declining_future.astype(int),time_scores)
comparison=pd.DataFrame({'split':['Naive random row split','Time-aware split'],'precision_at_50':[random_p50,time_p50],'evaluation_rows':[len(Xte),len(march)]})
display(comparison); print(f'Observed Precision@50 change: {time_p50-random_p50:+.3f}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,split,precision_at_50,evaluation_rows
0,Naive random row split,1.00,179399
1,Time-aware split,0.66,331436


Observed Precision@50 change: -0.340


## 3. Leakage audit

Final Week-5 features: impressions, clicks, CTR, average position, GA4 availability. The label uses only next-month impressions. IDs are for grouping/reporting, not features. Product decision flags and trend labels are excluded.

In [3]:
forbidden=['is_declining_future','next_imp','next_month','trend_direction','trend_pct','health_score','priority_score','action_type','refresh_tier','client_id','content_id']
audit=pd.DataFrame({'feature':features,'uses_future_or_decision_field':[f in forbidden for f in features]})
display(audit); print('Leakage status:', 'PASS' if not audit['uses_future_or_decision_field'].any() else 'REVIEW')

# Real error examples from the honest March test
err=march[['client_id','content_id','impressions','clicks','ctr','avg_position','is_declining_future']].copy(); err['model_score']=time_scores
top50=err.sort_values('model_score',ascending=False).head(50); fps=top50[top50.is_declining_future==0].head(5)
print(f'Top-50 false positives: {len(fps)}'); display(fps)

,feature,uses_future_or_decision_field
0,impressions,False
1,clicks,False
2,ctr,False
3,avg_position,False
4,ga4_available,False


Leakage status: PASS
Top-50 false positives: 5


,client_id,content_id,impressions,clicks,ctr,avg_position,is_declining_future,model_score
792356,client_62f4a7e64f5e0096,content_b2e02e58ec589f0e,46168.0,42.0,0.00091,8.428449,0.0,0.607207
853013,client_73cda7b4e4f265ea,content_d651a3a3f51d4ee8,7692.0,7.0,0.00091,4.562412,0.0,0.586722
724321,client_0fa64a184f18a4a0,content_9d0d97080c811f65,4.0,1.0,0.25000,4.166667,0.0,0.573995
828289,client_3ffa76342f366962,content_38f413bbbb6939f9,4.0,1.0,0.25000,4.333333,0.0,0.573995
727356,client_3ffa76342f366962,content_c90164af8319cbdd,4.0,1.0,0.25000,1.250000,0.0,0.572248


## 4. Claim rewrite

### Earlier / too-strong claim
‘The Random Forest predicts which pages should be refreshed and will improve their traffic.’

### Evidence-safe rewrite
‘On the March 2026 decision slice, the Random Forest ranked pages by a measured future-impression-decline proxy. Its Precision@50 is a directional decision-support metric for prioritizing human review; it does not show that refreshing a page will cause traffic to improve.’

The model predicts an observed future proxy, not the causal effect of a refresh. The time-aware result is evidence about this validation setup, not proof of equal generalization to every client or future period.

## Self-check

- [ ] Two paper findings and constructive methodology questions are included.
- [ ] Before/after random vs time-aware split is executed.
- [ ] Leakage audit and real pseudonymized error examples are included.
- [ ] Claims use observed/measured/directional/decision-support language.
- [ ] Run top-to-bottom in Colab, save the executed notebook, and commit it to `work/notebooks/w06_validation_audit.ipynb`.